# **Sistem Rekomendasi Destinasi Wisata di Indonesia Menggunakan Metode Content-Based Filtering**

Import Library

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords

# Exploratory Data Analysis

Data Load

In [ ]:
df = pd.read_csv('dataset/tourism_with_id.csv')
print("Informasi Dataset:")
df.info()

display(df.head(7))

Check Missing Value & Duplicate Data

In [ ]:
print("Jumlah Missing Values per Kolom:")
print(df.isnull().sum())

print("")

print("Jumlah baris duplikat: ")
print(df.duplicated().sum())

Distribusi Kategori

In [ ]:
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 6))
sns.countplot(y='Category', data=df, order=df['Category'].value_counts().index, palette='viridis')
plt.title('Distribusi Kategori Tempat Wisata di Indonesia', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Tempat Wisata', fontsize=12)
plt.ylabel('Kategori', fontsize=12)
plt.tight_layout()
plt.show()

Distribusi Kota

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(y='City', data=df, order=df['City'].value_counts().index, palette='magma')
plt.title('Distribusi Tempat Wisata Berdasarkan Kota', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah Tempat Wisata', fontsize=12)
plt.ylabel('Kota', fontsize=12)
plt.tight_layout()
plt.show()

Distribusi Rating dan Harga

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['Rating'], bins=20, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('Distribusi Nilai Rating', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frekuensi')

sns.histplot(df['Price'], bins=30, kde=True, ax=axes[1], color='salmon')
axes[1].set_title('Distribusi Harga Tiket Masuk (Price)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Harga (Rupiah)')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

Hubungan Kategori dengan Rating

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='Rating', y='Category', data=df, palette='Set2')
plt.title('Sebaran Rating Berdasarkan Kategori', fontsize=14, fontweight='bold')
plt.xlabel('Rating', fontsize=12)
plt.ylabel('Kategori', fontsize=12)
plt.tight_layout()
plt.show()

# Data Preprocessing

Data Cleaning & Feature Selection

In [ ]:
cols_to_drop = ['Time_Minutes', 'Coordinate', 'Lat', 'Long', 'Unnamed: 11', 'Unnamed: 12']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

print("Tampilan data setelah penghapusan kolom yang tidak relevan:")
display(df_clean.head(2))

df_clean = df_clean.dropna(subset=['Place_Name', 'Description', 'Category', 'City'])

Text Normalization

In [ ]:
import re

def text_cleaning(text):
    text = str(text)

    text = text.lower()
    
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    return text

df_clean['Desc_Clean'] = df_clean['Description'].apply(text_cleaning)
df_clean['Cat_Clean'] = df_clean['Category'].apply(text_cleaning)
df_clean['City_Clean'] = df_clean['City'].apply(text_cleaning)

print("Teks Sebelum Cleaning:")
print(df['Description'].iloc[0][:150] + "...")

print("\nTeks Setelah Cleaning:")
print(df_clean['Desc_Clean'].iloc[0][:150] + "...")

Feature Merging

In [ ]:
df_clean['tags'] = df_clean['Desc_Clean'] + " " + \
                   (df_clean['Cat_Clean'] + " ") * 2 + \
                   (df_clean['City_Clean'] + " ") * 2

data_preparation = df_clean[['Place_Id', 'Place_Name', 'Category', 'City', 'Rating', 'Price', 'tags']]

print("Dataset Final:")
display(data_preparation.head())

# (Opsional) Menyimpan data yang sudah bersih agar tidak perlu preprocessing ulang dari awal
# data_preparation.to_csv('tourism_cleaned.csv', index=False)

# Model

Baseline Model

In [ ]:
def get_popularity_baseline(dataframe, top_n=5):
    baseline = dataframe.sort_values(by=['Rating', 'Price'], ascending=[False, True])
    
    result = baseline[['Place_Id', 'Place_Name', 'Category', 'City', 'Rating', 'Price']].head(top_n)
    return result

print("Top 5 Destinasi Populer:")
display(get_popularity_baseline(df))

Model Content Based Filtering

In [ ]:
indo_stopwords = stopwords.words('indonesian')
custom_stopwords = indo_stopwords + ['wisata', 'tempat', 'berada', 'terletak', 'merupakan', 'juga', 'dengan']

tfidf = TfidfVectorizer(stop_words=custom_stopwords)

tfidf_matrix = tfidf.fit_transform(df_clean['tags'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Ukuran Matriks TF-IDF: {tfidf_matrix.shape}")
print(f"Ukuran Matriks Cosine Similarity: {cosine_sim.shape}")

# Recomendation Function

In [ ]:
def smart_recommender(keyword=None, kota=None, kategori=None, items=df_clean, similarity_data=cosine_sim, k=5):
    
    filtered_df = items.copy()
    
    if keyword:
        idx_list = items[items['Place_Name'].str.lower() == keyword.lower()].index
        
        if len(idx_list) > 0:
            idx = idx_list[0]
            sim_scores = list(enumerate(similarity_data[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:k+1]
            place_indices = [i[0] for i in sim_scores]
            
            recomends = items[['Place_Name', 'Category', 'City', 'Rating', 'Price', 'Description']].iloc[place_indices].copy()
            recomends['Similarity'] = [round(i[1], 2) for i in sim_scores]
            return recomends
        else:
            filtered_df = filtered_df[filtered_df['tags'].str.contains(keyword.lower(), na=False)]

    if kota:
        filtered_df = filtered_df[filtered_df['City'].str.lower() == kota.lower()]
    if kategori:
        filtered_df = filtered_df[filtered_df['Category'].str.lower() == kategori.lower()]

    if filtered_df.empty:
        return pd.DataFrame({"Pesan": ["Maaf, tidak ada wisata yang sesuai kriteria Anda."]})
    
    return filtered_df.sort_values(by=['Rating', 'Price'], ascending=[False, True])[['Place_Name', 'Category', 'City', 'Rating', 'Price', 'Description']].head(k)

# Evaluation Model

In [ ]:
def evaluate_model(target_place, k=5):
    target_data = df_clean[df_clean['Place_Name'] == target_place]
    if target_data.empty: return None
    
    target_category = target_data['Category'].values[0]
    
    recs = smart_recommender(keyword=target_place, k=k)
    
    total_relevant_in_dataset = len(df_clean[df_clean['Category'] == target_category]) - 1
    relevance_array = [1 if cat == target_category else 0 for cat in recs['Category']]
    
    hits = sum(relevance_array)
    precision_at_k = hits / k
    recall_at_k = hits / total_relevant_in_dataset if total_relevant_in_dataset > 0 else 0
    f1 = 2 * (precision_at_k * recall_at_k) / (precision_at_k + recall_at_k) if (precision_at_k + recall_at_k) > 0 else 0
        
    ap = 0
    relevant_count = 0
    for i, is_relevant in enumerate(relevance_array):
        if is_relevant == 1:
            relevant_count += 1
            ap += relevant_count / (i + 1)
            
    ap = ap / hits if hits > 0 else 0
    return precision_at_k, recall_at_k, f1, ap

def evaluate_all_dataset(k=5):
    precisions, recalls, f1_scores, ap_scores = [], [], [], []
    
    semua_tempat = df_clean['Place_Name'].tolist()
    
    for place in semua_tempat:
        hasil = evaluate_model(place, k=k)
        if hasil is None:
            continue
            
        p, r, f1, ap = hasil
        precisions.append(p)
        recalls.append(r)
        f1_scores.append(f1)
        ap_scores.append(ap)
        
    avg_precision = np.mean(precisions)
    avg_recall = np.mean(recalls)
    avg_f1 = np.mean(f1_scores)
    map_score = np.mean(ap_scores) 
    
    print("\n" + "="*40)
    print("  HASIL EVALUASI KESELURUHAN MODEL ")
    print("="*40)
    print(f"Total Data Dievaluasi : {len(precisions)} Tempat Wisata")
    print("-" * 40)
    print(f"Mean Precision@{k}   : {avg_precision:.4f}")
    print(f"Mean Recall@{k}      : {avg_recall:.4f}")
    print(f"Mean F1-Score       : {avg_f1:.4f}")
    print(f"MAP (Mean Avg Prec) : {map_score:.4f}")
    print("="*40)

evaluate_all_dataset(k=5)

# Testing

In [62]:
def interactive_user_testing():
    print("="*60)
    print("  SISTEM REKOMENDASI WISATA INDONESIA ")
    print("="*60)
    print("Petunjuk: Kosongkan jawaban (langsung tekan Enter) jika tidak ingin memfilter kriteria tersebut.")
    print("Ketik 'exit' atau 'keluar' pada Nama Tempat untuk menghentikan program.\n")
    
    while True:
        print("-" * 60)
        input_nama = input("1. Masukkan Nama Tempat / Keyword : ").strip()

        if input_nama.lower() in ['exit', 'keluar']:
            break
            
        input_kota = input("2. Masukkan Kota (Jakarta | Bandung | Semarang | Yogyakarta | Surabaya): ").strip()
        input_kategori = input("3. Masukkan Kategori (Taman Hiburan | Budaya | Cagar Alam | Bahari | Tempat Ibadah | Pusat Perbelanjaan): ").strip()

        keyword_param = input_nama if input_nama != "" else None
        kota_param = input_kota if input_kota != "" else None
        kategori_param = input_kategori if input_kategori != "" else None

        hasil = smart_recommender(keyword=keyword_param, kota=kota_param, kategori=kategori_param, k=5)

        print("\n HASIL REKOMENDASI ")
        display(hasil)
        print("\n")

interactive_user_testing()

  SISTEM REKOMENDASI WISATA INDONESIA 
Petunjuk: Kosongkan jawaban (langsung tekan Enter) jika tidak ingin memfilter kriteria tersebut.
Ketik 'exit' atau 'keluar' pada Nama Tempat untuk menghentikan program.

------------------------------------------------------------

 HASIL REKOMENDASI 


,Place_Name,Category,City,Rating,Price,Description,Similarity
5,Taman Impian Jaya Ancol,Taman Hiburan,Jakarta,4.5,25000,Taman Impian Jaya Ancol merupakan sebuah objek...,0.61
8,Pelabuhan Marina,Bahari,Jakarta,4.4,175000,Pelabuhan Marina Ancol berada di kawasan Taman...,0.25
3,Taman Mini Indonesia Indah (TMII),Taman Hiburan,Jakarta,4.5,10000,Taman Mini Indonesia Indah merupakan suatu kaw...,0.25
46,Taman Situ Lembang,Taman Hiburan,Jakarta,4.5,0,Taman Situ Lembang adalah sebuah taman kota ya...,0.22
78,Taman Spathodea,Taman Hiburan,Jakarta,4.6,0,Objek Wisata Taman Spathodea di Jagakarsa DKI ...,0.21




------------------------------------------------------------

 HASIL REKOMENDASI 


,Place_Name,Category,City,Rating,Price,Description
401,Food Junction Grand Pakuwon,Pusat Perbelanjaan,Surabaya,4.5,0,Food Junction Grand Pakuwon sebetulnya merupak...




------------------------------------------------------------

 HASIL REKOMENDASI 


,Place_Name,Category,City,Rating,Price,Description
167,Pantai Timang,Bahari,Yogyakarta,4.7,10000,Pantai Timang adalah objek wisata berupa panta...
154,Pantai Ngobaran,Bahari,Yogyakarta,4.6,5000,Pantai Ngobaran merupakan salah satu tempat wi...
196,Pantai Jungwok,Bahari,Yogyakarta,4.6,5000,Pantai Jungwok adalah pantai yang terletak di ...
197,Pantai Greweng,Bahari,Yogyakarta,4.6,5000,"Di Kabupaten Gunungkidul, tidak sulit memilih ..."
199,Pantai Watu Kodok,Bahari,Yogyakarta,4.6,5000,Pantai Watu Kodok merupakan salah satu pantai ...




------------------------------------------------------------
